# SB3 DQN vs FHR-regularised SB3 DQN — MountainCar

Stable-Baselines3's best published sample-efficient DQN-family setting for this
env — **RL-Zoo-tuned DQN** (this config runs `algo.type: dqn`; switch it to
`qrdqn` for the `sb3_contrib` QR-DQN variant, the zoo tuning is identical apart
from `n_quantiles`)
— versus the same method with the **FHR learned-recurrence penalty**
bootstrapped onto its TD loss (`src/agents/sb3_fhr.py`). One config
(`configs/config_sb3.yaml`), one code path (`FHRDQN`), one difference per
arm: the **baseline arm** strips the FHR term (`fhr_weight: 0.0`, bit-for-bit
the stock SB3 algorithm — asserted in `tests/test_sb3_fhr.py`); each
**`exp<N>` arm** applies one FHR parameter set from
`experiment.fhr_experiments` on top of the shared `agent:` block. Every arm
trains once per seed in `experiment.seeds`, fanned out as parallel
subprocesses by `../src/run_sb3_seeds.py`, and every comparison figure
reports the seed-average.

The constant `-1`/step reward makes the exact Bellman recurrence roots {1, 1/gamma}; the classic MountainCar experiments found the on-policy Hankel rank of the value sequences to be ~2, the FHR sweet spot.

SB3 trains in gradient bursts (`train_freq`/`gradient_steps`), so
`train_diagnostics.csv` holds one row per `train()` burst — the loader below
aggregates rows to per-episode means. The low-rank Hankel sweep, the Q-matrix
rank tracking and the autoregressive value probe are ON by default
(`analysis:` block), dispatched through the same `training.run_analysis_tick`
the classic runs use, so every run records the full mechanism diagnostics —
re-runnable post-hoc on any checkpoint with `../src/run_sb3_analysis.py`.

All artifacts land under `cached/runs/<name>_<timestamp>/` with the manifest
`cached/sb3_runs_manifest.json` and are browsable in the web app:
`python result_viewer_app/rank_viewer.py` → compare mode shows baseline vs
each `exp<N>` as variants of the `sb3_runs` family.

## 0 · Launch - train whatever the config defines

In [ ]:
import pathlib, sys
SRC_RUNNERS = pathlib.Path.cwd().parent / "src"
if str(SRC_RUNNERS) not in sys.path:
    sys.path.insert(0, str(SRC_RUNNERS))
import run_sb3_seeds as runner
import yaml

CONFIG = "configs/config_sb3.yaml"
MANIFEST = "cached/sb3_runs_manifest.json"
# Every arm below - launched and analysed - is exactly what the config's
# experiment.fhr_experiments block currently defines; nothing is hardcoded.
EXPERIMENTS = sorted(int(k) for k in
                     (yaml.safe_load(open(CONFIG))["experiment"]
                      .get("fhr_experiments") or {}))
print("config experiments:", EXPERIMENTS)

LAUNCH = False
FORCE_EXP = False
if LAUNCH:
    manifest = runner.launch_all(config=CONFIG, experiments=EXPERIMENTS,
                                 max_workers=14, force=FORCE_EXP)
    print(sorted(manifest["runs"]))
else:
    print("LAUNCH = False - analysing existing runs only")

## Setup - the figure toolkit

Every figure in this notebook comes from
[`analysis.visualisations.fhr_figures`](../../../src/analysis/visualisations/fhr_figures.py),
so the four MuJoCo notebooks plot the same family the same way and a fix lands
once. Three conventions matter for reading the plots:

* **Colour identifies the arm.** Not lambda - an earlier version coloured by
  lambda, which painted every curve of a single-lambda grid (Ant's tuned family
  is entirely lambda = 0.1) the same blue. Arms take successive colours from the
  Okabe-Ito colour-blind-safe palette; the baseline is always near-black;
  **dashed = frozen-c control**, solid = learned c.
* **The seed band is mean +- 1 s.e.m.**, not the min-max envelope. Switch with
  `ff.BAND = "ci95" | "iqr" | "minmax"` *before* `load_family`.
* **Every `fig_*` call returns one standalone figure**, saved by `F.save(...)`
  to `figures/<family>/<name>.pdf` and `.png` at 300 dpi with Type-42 fonts -
  drop the PDF straight into the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "analysis").is_dir())
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from analysis.visualisations import fhr_figures as ff

ff.set_pub_style()
ff.BAND = "sem"          # seed band: "sem" | "ci95" | "iqr" | "minmax"

F = ff.load_family(CONFIG, MANIFEST)
F.summary()

# The sample-efficiency ladder. Read off the TRAINING stream (section 1), not
# the greedy-eval curve - see section 3. Automatic 1-2-5 ladder; pass step=/start= to pin it.
THRESHOLDS = F.auto_thresholds()
print("\nthresholds:", THRESHOLDS)

## 1 · Training curves - the episodes as trained

`rewards.csv`: every training episode's return exactly as the stochastic policy
experienced it (SAC's sampled actions, exploration included), against cumulative
env steps, rolling-mean smoothed. One figure per arm against the shared
baseline, so each panel is a clean two-curve comparison you can paste on its
own. Dots mark where the seed-mean curve first crosses each threshold of the
sample-efficiency ladder; the same crossings are tabulated in section 3.

This is the noisy, un-paired counterpart of section 2's greedy-eval curves -
the gap between the two is the exploration cost, and early-terminated episodes
enter at their actual (short) length.

In [ ]:
# One figure per FHR arm vs the baseline, each saved separately.
for a in F.fhr_arms:
    fig = F.fig_training(a.key, thresholds=THRESHOLDS)
    F.save(fig, f"01_training_{a.key}")
    plt.show()

In [ ]:
# ... and the overlay, for the single-figure version of the same story.
fig = F.fig_overlay("train")
F.save(fig, "01_training_all")
plt.show()

## 2 · Learning curves - greedy evaluation

`eval.csv`: the deterministic policy on fixed reset seeds, so the curves are
paired across arms and the training stream is untouched. Again one figure per
arm against the baseline, then the overlay.

In [ ]:
for a in F.fhr_arms:
    fig = F.fig_eval(a.key)
    F.save(fig, f"02_eval_{a.key}")
    plt.show()

In [ ]:
fig = F.fig_overlay("eval")
F.save(fig, "02_eval_all")
plt.show()

## 3 · Sample efficiency - the claim FHR actually makes

Every prior FHR result in this repo is about **onset / sample efficiency**, not
asymptote. The measurement here is deliberately taken on the **training
stream** (section 1) rather than the greedy-eval curve: the training curve is
what the agent's own experience looks like, it is sampled every episode instead
of every 20k steps, and it is the stream a sample-efficiency claim is about.

For each arm and each threshold we take the first env step at which that seed's
rolling-mean training return reaches it, then average over seeds. A threshold
that only some seeds ever reach is marked - averaging the ones that made it
biases the number downwards, so those points are drawn as open markers and left
off the line rather than allowed to bend it.

In [ ]:
_ = F.table_sample_efficiency(THRESHOLDS)

In [ ]:
fig = F.fig_steps_to_threshold(THRESHOLDS)
F.save(fig, "03_steps_to_threshold")
plt.show()

In [ ]:
# The same numbers as a ratio: > 1 means the arm reached that return in fewer
# environment steps than the baseline.
fig = F.fig_speedup(THRESHOLDS)
F.save(fig, "03_speedup")
plt.show()

## 4 · Final performance - one figure per lambda

Final greedy-eval return, one figure per lambda rung so each is a self-contained
arms-vs-baseline comparison. Bar = seed mean, whisker = +- 1 s.e.m., open dots =
the individual seeds (they are the honest picture of spread on 5 seeds), and the
percentage is the change against the baseline of the same figure.

In [ ]:
for lam in F.lambdas:
    fig = F.fig_final(lam)
    F.save(fig, f"04_final_lambda{lam:g}")
    plt.show()

In [ ]:
# The whole learned-c sweep as one lambda x order map (skipped when the family
# has only one lambda or one order - there is no grid to draw).
if len(F.lambdas) > 1 and len(F.orders) > 1:
    fig = F.fig_grid()
    F.save(fig, "04_grid")
    plt.show()

## 5 · FHR + SAC internals

`train_diagnostics.csv` is one row per gradient step - about 1e6 rows and 180 MB
per run, and the previous version of this notebook re-parsed all of it with
`csv.DictReader` for every one of nine panels. `F.prime_diag_cache()` reads each
file **once** with pandas, reduces it to a per-bin median of every column, gives
it an env-step axis and caches the result to `<run>/diag_binned_600.npz` (about
27 kB). The first call costs a few seconds per run; every call after is
instant, and the panels are a few hundred points per curve instead of a million,
so they render and re-render immediately.

The panel that decides whether the lambda ladder was placed correctly is
**rho = lambda·penalty / TD loss** - the fraction of the critic objective the
recurrence term actually owns. fetch_reach's calibrated pipeline targets
rho in [0.5, 2]; read off which rung of the ladder landed in that band.

In [ ]:
F.prime_diag_cache()      # first call: a few seconds per run. Then cached.
fig = F.fig_internals()
F.save(fig, "05_internals")
plt.show()

In [ ]:
F.table_rho()

## 6 · Hankel low-rankness of the value trajectory

The discrete-action counterpart of the rollout Hankel: roll the trained policy
greedily, Hankel the min-twin `Q(s_t, a_t)` sequence and compare spectra. The
claim under test is that the greedy rollout's Q sequence obeys a low-order
linear recurrence, so FHR arms should concentrate the spectrum faster than the
baseline.

This is a *rollout* measurement. The next section measures rank where the
penalty is actually applied instead.

In [ ]:
fig, table = F.fig_value_hankel(runner)
F.save(fig, "06_value_hankel")
plt.show()
print(f"{'arm':30s} rank@99.9%")
for label, r in table:
    print(f"{label:30s} {r:10d}")

## 7 · Penalised-window Hankel rank - the in-training probe

Rank measured **where the penalty is applied**: on sampled replay windows
(anchor + `window_rank_lags` same-episode predecessors, online critics, buffer
actions) rather than on greedy on-policy rollouts. The probe runs for every arm
**including the lambda = 0 baseline**, which samples and measures the same
windows, so its curve is the control: if FHR operates as a rank constraint, its
arms should push the window rank and the penalty-block tail ratio *below* the
baseline on exactly these windows.

The probe is an **in-training** measurement - it cannot be back-filled from
finished runs. It writes `window_hankel.csv` only when `agent.window_rank_every
> 0` was set in the config the runs were launched from; `F.window_probe_status()`
below says whether this family has it.

In [ ]:
_ = F.window_probe_status()

In [ ]:
if F.has_window_probe:
    for a in F.fhr_arms:
        fig = F.fig_window_rank(a.key)
        F.save(fig, f"07_window_rank_{a.key}")
        plt.show()
    fig = F.fig_window_rank_overlay()
    F.save(fig, "07_window_rank_all")
    plt.show()
    print()
    F.table_window_rank()

## 8 · Compute cost

In [ ]:
import os
rows = []
for k, a in F.arms.items():
    for s, d in F.run_dirs(k):
        ck = d / "checkpoints" / "final.pt"
        if ck.exists():
            rows.append((a.plain, s, (os.path.getmtime(ck)
                                      - os.path.getmtime(d / "config.yaml")) / 60))
for label, s, mins in rows:
    print(f"{label:30s} seed {s}: {mins:6.1f} min")
base_name = F.baseline.plain if F.baseline else None
base = [m for l, _, m in rows if l == base_name]
fhr = [m for l, _, m in rows if l != base_name]
if base and fhr:
    print(f"\nbaseline mean {np.mean(base):.1f} min; FHR-arm mean "
          f"{np.mean(fhr):.1f} min (overhead x{np.mean(fhr)/np.mean(base):.2f})")

## 9 · Findings

*(fill in after the sweep completes)*

## 10 · Bespoke analysis kept from the previous version of this notebook

The cells below are this study's own analyses - they are not part of the shared
figure toolkit and were left exactly as they were. `F.legacy_namespace()`
rebuilds the names they expect (`ARMS`, `INFO`, `BASE`, `GLOBALS_`, `VARIANTS`,
`PER_ARMS`, `labels_of`, `run_dirs`, `curves`, `diag`) from the `Family`, so
they keep running **and** pick up the new per-arm colours, since `ARMS` carries
them.

In [ ]:
import csv, glob, json
try:
    import torch
except ImportError:
    torch = None
globals().update(F.legacy_namespace())
print("legacy names restored:", ", ".join(sorted(F.legacy_namespace())))
if not F.arms:
    print()
    print("*** NO RUNS FOR THIS FAMILY ON THIS MACHINE ***")
    print("The cells below are this study's own code and need its run")
    print("directories under cached/runs/. Everything above degrades to an")
    print("empty-family notice; the cells below will raise instead.")

In [ ]:
import csv, sys, pathlib, warnings
import matplotlib.pyplot as plt
import numpy as np
import yaml

# NaN-padded seed stacks legitimately have all-NaN columns (episodes before
# training starts / after early-stopped seeds end) — nanmean warns, we don't care.
warnings.filterwarnings("ignore", message="Mean of empty slice")

EXPS = pathlib.Path.cwd().parent / "src"   # stable_baselines_3/src — hosts the shared runner
if str(EXPS) not in sys.path:
    sys.path.insert(0, str(EXPS))

from run_sb3_seeds import launch_all, load_runs, CONFIG

### All arms, all seeds — parallel subprocess fan-out

One training subprocess per (arm, seed): the shared baseline (`fhr_weight: 0.0`
— the single difference, reused from the manifest if already trained) plus one
`exp<N>` arm per entry of the config's `experiment.fhr_experiments`. Completed
runs are recorded in `cached/sb3_runs_manifest.json` — re-running the cell only
launches missing or failed pairs — and each child logs to
`cached/logs/sb3_<arm>_seed<N>.log`.

In [ ]:
# Train the shared baseline + every numbered experiment defined in the config.
with open(CONFIG) as f:
    EXP_NUMS = sorted(yaml.safe_load(f)["experiment"].get("fhr_experiments") or {})
print(f"experiments in {CONFIG}: {EXP_NUMS}")
if LAUNCH:                     # set in the launch cell at the top
    manifest = launch_all(max_workers=6, experiments=EXP_NUMS)
else:
    manifest = (json.load(open(MANIFEST)) if pathlib.Path(MANIFEST).exists()
                else {"seeds": [], "runs": {}})
    print("LAUNCH = False - analysing existing runs only")
SEEDS = manifest["seeds"]
runs_base = load_runs("baseline")
[r["run_dir"] for r in runs_base]

### Numbered experiment arms

Loaded from the same manifest. Each arm's FHR parameter set is its
manifest-recorded `agent_overrides` applied on top of the shared `agent:` block
— the `config.yaml` copy inside a run dir does **not** reflect the overrides,
so `agent_by_arm` below is the effective configuration each arm actually
trained with.

In [ ]:
ARMS_FHR = [f"exp{n}" for n in EXP_NUMS]
runs_by_arm = {arm: load_runs(arm) for arm in ARMS_FHR}
ALL_ARMS = [("baseline", runs_base)] + [(arm, runs_by_arm[arm]) for arm in ARMS_FHR]

# effective FHR params per arm = shared agent block + the manifest overrides
agent_by_arm = {arm: {**rs[0]["cfg"]["agent"], **(rs[0]["agent_overrides"] or {})}
                for arm, rs in runs_by_arm.items()}
cfg_shared = runs_base[0]["cfg"]   # non-FHR blocks are identical across arms

_PALETTE = ["indianred", "darkorange", "mediumpurple", "seagreen", "goldenrod"]
ARM_COLORS = {"baseline": "steelblue",
              **{arm: _PALETTE[i % len(_PALETTE)] for i, arm in enumerate(ARMS_FHR)}}

def arm_label(arm):
    if arm == "baseline":
        return "baseline (stock SB3, fhr_weight = 0)"
    a = agent_by_arm[arm]
    arx = ", ARX" if a.get("reward_lags") else ""
    return f"{arm} (lambda={a['fhr_weight']}, r={a['fhr_order']}{arx})"

{arm: rs[0]["agent_overrides"] for arm, rs in runs_by_arm.items()}

### Comparison figures

All curves below are seed-averages (thin lines, where shown, are single seeds).
Each figure is shown inline and saved as `figures/comparison_<slug>.png` into
**every** run directory (all arms, all seeds), so any run's page in the result
viewer carries the full comparison. Diagnostics rows (one per SB3 `train()`
burst) are aggregated to per-episode means before the seed-average.

In [ ]:
def read_diagnostics(run_dir):
    """train_diagnostics.csv -> {column: np.array} aggregated to one value
    per episode (SB3 logs one row per train() burst; several bursts can share
    an episode index, so rows are nanmean-averaged per episode)."""
    path = pathlib.Path(run_dir) / "train_diagnostics.csv"
    with open(path) as f:
        rows = list(csv.DictReader(f))
    eps = sorted({int(float(r["episode"])) for r in rows})
    out = {"episode": np.array(eps, dtype=float)}
    for k in rows[0]:
        if k == "episode":
            continue
        by_ep = {}
        for r in rows:
            by_ep.setdefault(int(float(r["episode"])), []).append(float(r[k]))
        out[k] = np.array([np.nanmean(by_ep[e]) if by_ep[e] else np.nan for e in eps])
    return out

diags_base = [read_diagnostics(r["run_dir"]) for r in runs_base]
diags_by_arm = {arm: [read_diagnostics(r["run_dir"]) for r in rs]
                for arm, rs in runs_by_arm.items()}
ALL_DIAGS = [("baseline", diags_base)] + [(a, diags_by_arm[a]) for a in ARMS_FHR]

ALL_RUN_DIRS = [r["run_dir"] for _, runs in ALL_ARMS for r in runs]

def save_and_show(fig, slug):
    for d in ALL_RUN_DIRS:
        figdir = pathlib.Path(d) / "figures"
        figdir.mkdir(exist_ok=True)
        fig.savefig(figdir / f"comparison_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()

def rolling(x, w=50):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

def stack_padded(seqs):
    """(n_seeds, max_len) array, NaN-padded on the right — early-stopped seeds
    are shorter, and the NaN-aware means below only average seeds still running."""
    out = np.full((len(seqs), max(len(s) for s in seqs)), np.nan)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = s
    return out

def diag_seed_mean(diags, key):
    """(episode axis, seed-mean curve) for one diagnostics column, aligned on
    the integer episode index and NaN-padded per seed."""
    n_ep = int(max(d["episode"].max() for d in diags)) + 1
    stack = np.full((len(diags), n_ep), np.nan)
    for i, d in enumerate(diags):
        stack[i, d["episode"].astype(int)] = d[key]
    return np.arange(n_ep), np.nanmean(stack, axis=0)

def episode_steps(r):
    """Per-episode env-step counts: the rewards.csv steps column when the run
    has one, else |reward| — within one step of exact here: MountainCar pays -1 every step."""
    if r.get("steps") is not None:
        return np.asarray(r["steps"], dtype=float)
    return np.abs(np.asarray(r["rewards"], dtype=float))

def env_steps_axis(r):
    """Cumulative env steps at the end of each episode — the samples x-axis
    learning curves are reported on in the literature."""
    return np.cumsum(episode_steps(r))

def seed_mean_on_steps(runs, ys, n=400):
    """Seed-average curves whose x values (env steps) don't align across seeds:
    linearly interpolate each seed's (env_steps, y) curve onto a shared grid,
    aggregating only the seeds whose own span covers each grid point — an
    early-stopped seed drops out of the mean instead of being extrapolated.
    ys[i] aligns with the END of runs[i]'s episodes (rolling() output is fine).
    Returns (grid, mean, min, max)."""
    curves = []
    for r, y in zip(runs, ys):
        y = np.asarray(y, dtype=float)
        x = env_steps_axis(r)[-len(y):]
        curves.append((x, y))
    lo = min(float(x[0]) for x, _ in curves)
    hi = max(float(x[-1]) for x, _ in curves)
    grid = np.linspace(lo, hi, n)
    stack = np.full((len(curves), n), np.nan)
    for i, (x, y) in enumerate(curves):
        inside = (grid >= x[0]) & (grid <= x[-1])
        stack[i, inside] = np.interp(grid[inside], x, y)
    return (grid, np.nanmean(stack, axis=0),
            np.nanmin(stack, axis=0), np.nanmax(stack, axis=0))

In [ ]:
# -- learning curves (x = environment steps, i.e. samples collected) ----------
W = 50
fig, ax = plt.subplots(figsize=(9, 4.5))
for arm, runs in ALL_ARMS:
    for r in runs:
        ax.plot(env_steps_axis(r), r["rewards"], alpha=0.12, color=ARM_COLORS[arm])
for arm, runs in ALL_ARMS:
    grid, m, lo, hi = seed_mean_on_steps(runs, [rolling(r["rewards"], W) for r in runs])
    ax.plot(grid, m, color=ARM_COLORS[arm],
            label=f"{arm_label(arm)}, {len(SEEDS)}-seed mean")
    ax.fill_between(grid, lo, hi, color=ARM_COLORS[arm], alpha=0.15)
ax.axhline(cfg_shared["training"]["solved_reward"], ls="--", c="gray", lw=1, label="solved")
ax.set_xlabel("environment steps (samples)")
ax.set_ylabel("episode return")
ax.set_title(f"Learning curves (thin = seeds, thick = rolling-{W} seed mean, band = min-max)")
ax.legend()
fig.tight_layout()
save_and_show(fig, "learning_curves")

### Reference: the standard RL-Zoo baseline

The tuned recipe (Polyak targets, lr 1e-3 — see
`../mountaincar_tuning/TUNING_LOG.md`) changed the *base config under both
arms*, so the `lambda=0` arm above is the scientifically **paired** baseline.
This section adds the **published** reference: the stock RL-Zoo recipe from
`configs/config_sb3_zoo.yaml` (published zoo result: −100.85 ± 9.9,
deterministic eval), launched into its own manifest via
`launch_all(config=..., arms=["baseline"])` so the two recipes can share this
directory. Cross-recipe curves are compared on greedy eval (`eval.csv`), not
training reward.

In [ ]:
# -- tuned recipe vs the standard RL-Zoo baseline (greedy eval) ---------------
# configs/config_sb3_zoo.yaml = the stock zoo recipe (hard sync/600, lr 4e-3),
# run as an external reference. Its runs land in their own manifest
# (cached/sb3_runs_manifest_zoo.json), so they never collide with the tuned
# arms above; re-running the cell only launches missing pairs. NOTE: first
# launch = 5 baseline runs with full analysis ticks (~50 min each, 5 in
# parallel) — set analysis methods: [] / enabled: false in the zoo config for
# fast reward-only runs.
ZOO_CONFIG = "configs/config_sb3_zoo.yaml"
launch_all(max_workers=6, config=ZOO_CONFIG, arms=["baseline"])
runs_zoo = load_runs("baseline", config=ZOO_CONFIG)

def eval_curve(run_dir):
    """eval.csv -> (env_steps, greedy mean reward). The deterministic-eval
    curve is the only measure comparable across recipes whose exploration
    floors differ (tuned and zoo both train with eps_final 0.07, but any
    cross-recipe claim should not ride on the training-reward noise)."""
    t = np.genfromtxt(pathlib.Path(run_dir) / "eval.csv", delimiter=",",
                      names=True, ndmin=1)
    return t["env_steps"], t["mean_reward"]

BEST_FHR = ARMS_FHR[1]          # exp10 = the tuned lambda .5, r=2 headline arm
GRID = np.arange(0, 150001, 5000)
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, runs, color in [
        (f"tuned {arm_label(BEST_FHR)}", runs_by_arm[BEST_FHR], ARM_COLORS[BEST_FHR]),
        ("tuned baseline (lambda=0)", runs_base, "steelblue"),
        ("standard zoo baseline (hard sync/600, lr 4e-3)", runs_zoo, "dimgray")]:
    curves = [eval_curve(r["run_dir"]) for r in runs]
    for s, v in curves:
        ax.plot(s, v, color=color, alpha=0.15)
    # np.interp holds an early-stopped (solved) seed's last eval value forward
    m = np.mean([np.interp(GRID, s, v) for s, v in curves], axis=0)
    ax.plot(GRID, m, color=color, label=f"{label}, {len(curves)}-seed mean")
ax.axhline(-120, ls="--", c="gray", lw=1, label="solved (greedy)")
ax.set_xlabel("environment steps (samples)")
ax.set_ylabel("greedy-eval reward (10 eps, fixed reset seeds)")
ax.set_title("Hankel Regularised DQN vs RL-Zoo baseline")
ax.legend()
fig.tight_layout()
save_and_show(fig, "zoo_reference_eval")

In [ ]:
# -- TD error over training --------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
for arm, diags in ALL_DIAGS:
    ax.plot(*diag_seed_mean(diags, "td_loss"), color=ARM_COLORS[arm], alpha=0.8,
            label=arm_label(arm))
ax.set_xlabel("episode"); ax.set_ylabel("TD loss (per-episode mean)")
ax.set_yscale("log")
ax.set_title(f"TD error over training ({len(SEEDS)}-seed mean)")
ax.legend()
save_and_show(fig, "td_error")

In [ ]:
# -- regularisation penalty over training (seed means) -----------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax2 = axes[1].twinx()
for arm in ARMS_FHR:
    diags, color = diags_by_arm[arm], ARM_COLORS[arm]
    axes[0].plot(*diag_seed_mean(diags, "penalty_raw"), color=color, label=arm)
    axes[1].plot(*diag_seed_mean(diags, "penalty_weighted"), color=color)
    ax2.plot(*diag_seed_mean(diags, "lambda_eff"), color=color, ls="--", alpha=0.5)
    axes[2].plot(*diag_seed_mean(diags, "residual_rms"), color=color)
axes[0].set_title("penalty_raw (unweighted recurrence residual)")
axes[0].legend()
axes[1].set_title("penalty_weighted = lambda_eff x penalty_raw")
ax2.set_ylabel("lambda_eff (dashed)", color="gray")
axes[2].set_title("residual RMS")
for ax in axes:
    ax.set_xlabel("episode")
fig.suptitle(f"FHR penalty over training ({len(SEEDS)}-seed mean; hard warm-up: lambda engages at full strength)")
fig.tight_layout()
save_and_show(fig, "penalty")

In [ ]:
# -- learned recurrence coefficients (one panel per experiment arm) -----------
gamma = cfg_shared["algo"]["gamma"]
c_d_cols = {}   # per-arm coefficient columns, reused by the summary cell
fig, axes = plt.subplots(1, len(ARMS_FHR), figsize=(5 * len(ARMS_FHR), 4),
                         squeeze=False)
for ax, arm in zip(axes[0], ARMS_FHR):
    diags, order = diags_by_arm[arm], agent_by_arm[arm]["fhr_order"]
    c_cols = [f"c_{j}" for j in range(1, order + 1) if f"c_{j}" in diags[0]]
    d_cols = [f"d_{j}" for j in range(1, order + 1) if f"d_{j}" in diags[0]]
    c_d_cols[arm] = c_cols + d_cols
    for col in c_cols + d_cols:
        ax.plot(*diag_seed_mean(diags, col), label=col)
    ax.axhline(1 + 1 / gamma, ls=":", c="gray", lw=1)
    ax.axhline(-1 / gamma, ls=":", c="gray", lw=1)
    ax.set_title(arm_label(arm))
    ax.set_xlabel("episode")
    ax.legend()
fig.suptitle("Learned recurrence coefficients (seed mean; dotted: constant-reward Bellman values)")
fig.tight_layout()
save_and_show(fig, "coefficients")

# -- recurrence health: unit root + companion spectral radius -----------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for arm in ARMS_FHR:
    axes[0].plot(*diag_seed_mean(diags_by_arm[arm], "sum_c"),
                 color=ARM_COLORS[arm], label=arm)
    axes[1].plot(*diag_seed_mean(diags_by_arm[arm], "companion_radius"),
                 color=ARM_COLORS[arm])
axes[0].axhline(1.0, ls=":", c="gray", lw=1)
axes[0].set_title("sum_c (1 = unit root of the Bellman recurrence)")
axes[0].legend()
axes[1].axhline(1 / gamma, ls=":", c="gray", lw=1)
axes[1].set_title("companion spectral radius (reference: 1/gamma)")
for ax in axes:
    ax.set_xlabel("episode")
fig.suptitle(f"Learned recurrence over training ({len(SEEDS)}-seed mean)")
fig.tight_layout()
save_and_show(fig, "recurrence_health")

In [ ]:
# -- penalty batch composition (seed means) ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for arm in ARMS_FHR:
    axes[0].plot(*diag_seed_mean(diags_by_arm[arm], "b_h"),
                 color=ARM_COLORS[arm], label=arm)
    axes[1].plot(*diag_seed_mean(diags_by_arm[arm], "unique_eps"),
                 color=ARM_COLORS[arm])
axes[0].set_title("b_h: samples with r same-episode predecessors (of batch %d)"
                  % cfg_shared["algo"]["batch_size"])
axes[0].legend()
axes[1].set_title("unique episodes contributing penalty samples")
for ax in axes:
    ax.set_xlabel("episode")
fig.tight_layout()
save_and_show(fig, "penalty_batch")

In [ ]:
# -- summary -----------------------------------------------------------------
def solve_episode(rewards, solved, patience=50):
    roll = rolling(rewards, patience)
    idx = np.argmax(roll > solved)
    return int(idx + patience) if roll.max() > solved else None

solved = cfg_shared["training"]["solved_reward"]
for arm, runs in ALL_ARMS:
    bests = [rolling(r["rewards"]).max() for r in runs]
    solves = [solve_episode(r["rewards"], solved) for r in runs]
    # env steps consumed up to the solve episode — the sample-complexity number
    samples = [int(env_steps_axis(r)[s - 1]) if s is not None else None
               for r, s in zip(runs, solves)]
    solved_eps = [s for s in solves if s is not None]
    solved_smp = [s for s in samples if s is not None]
    mean_solve = f"{np.mean(solved_eps):.0f}" if solved_eps else "n/a"
    mean_smp = f"{np.mean(solved_smp):,.0f}" if solved_smp else "n/a"
    print(f"{arm_label(arm)}: mean best rolling-50 {np.mean(bests):.1f} | "
          f"solved {len(solved_eps)}/{len(runs)} seeds | mean solve episode {mean_solve} "
          f"| mean env steps to solve {mean_smp}")
    for r, b, s, smp in zip(runs, bests, solves, samples):
        total = int(env_steps_axis(r)[-1])
        print(f"    seed {r['seed']}: {len(r['rewards'])} episodes / {total:,} env steps, "
              f"best rolling-50 {b:.1f}, solved at episode {s}"
              + (f" ({smp:,} env steps)" if smp is not None else ""))

for arm in ARMS_FHR:
    final = {c: np.mean([d[c][-1] for d in diags_by_arm[arm]])
             for c in c_d_cols[arm] + ["sum_c", "companion_radius"]}
    print(f"{arm} final coefficients (seed mean): "
          + ", ".join(f"{c}={final[c]:.4f}" for c in c_d_cols[arm])
          + f" | sum_c={final['sum_c']:.4f}"
          + f" | companion radius={final['companion_radius']:.4f} (1/gamma={1/gamma:.4f})")
print("nan_skips: " + " | ".join(
    arm + " " + ", ".join(f"{d['nan_skips'][-1]:.0f}" for d in diags)
    for arm, diags in ALL_DIAGS))

### Mechanism: low-rank structure over training (seed-averaged)

Every run records the greedy-rollout Hankel sweep (`hankel_sweep.csv`) and the
state-grid Q-matrix rank (`rank_stats.csv`) every `analysis.ep_freq` episodes —
these curves are the mechanism claim: does the FHR penalty pull the
value-sequence Hankel toward the target rank faster than the stock SB3
baseline? Per-episode spectra figures live in each run's `figures/`
(browsable in the result viewer).

In [ ]:
# -- Hankel effective rank over training (thin = seeds, thick = mean) ---------
def rank_rows(run_dir, csv_name, matrix, col="eff_rank"):
    """{episode: mean of `col` over rollouts} for one run."""
    with open(pathlib.Path(run_dir) / csv_name) as f:
        rows = [r for r in csv.DictReader(f) if r["matrix"] == matrix]
    eps = sorted({int(r["episode"]) for r in rows})
    return {e: np.mean([float(r[col]) for r in rows if int(r["episode"]) == e])
            for e in eps}

def arm_rank_stack(runs, csv_name, matrix, col="eff_rank"):
    """(episode axis, (n_seeds, n_ticks) NaN-padded stack) across an arm."""
    per = [rank_rows(r["run_dir"], csv_name, matrix, col) for r in runs]
    all_eps = sorted(set().union(*per))
    stack = np.array([[p.get(e, np.nan) for e in all_eps] for p in per], float)
    return np.array(all_eps), stack

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, matrix in zip(axes, ("Hankel Q", "Hankel V")):
    for arm, runs in ALL_ARMS:
        eps, stack = arm_rank_stack(runs, "hankel_sweep.csv", matrix)
        for row in stack:
            ax.plot(eps, row, color=ARM_COLORS[arm], alpha=0.2)
        ax.plot(eps, np.nanmean(stack, axis=0), color=ARM_COLORS[arm], label=arm)
    for order in sorted({agent_by_arm[a]["fhr_order"] for a in ARMS_FHR}):
        ax.axhline(order, ls=":", c="gray", lw=1)
    ax.set_title(f"{matrix} effective rank (dotted: fhr_order targets)")
    ax.set_xlabel("episode")
axes[0].set_ylabel("effective rank (greedy rollouts)")
axes[0].legend()
fig.tight_layout()
save_and_show(fig, "hankel_rank_evolution")

In [ ]:
# -- state-grid Q-matrix rank over training ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ("eff_rank", "stable_rank")):
    for arm, runs in ALL_ARMS:
        eps, stack = arm_rank_stack(runs, "rank_stats.csv", "Q-function", col)
        for row in stack:
            ax.plot(eps, row, color=ARM_COLORS[arm], alpha=0.2)
        ax.plot(eps, np.nanmean(stack, axis=0), color=ARM_COLORS[arm], label=arm)
    ax.set_title(f"state-grid Q-matrix {col}")
    ax.set_xlabel("episode")
axes[0].legend()
fig.tight_layout()
save_and_show(fig, "q_matrix_rank")

### Final greedy policies (video)

One greedy episode per (arm, seed), rolled out from `checkpoints/final.pt`
(the SB3 checkpoint zip carries the FHR coefficients itself) and reset with
the run's own seed; saved to `<run_dir>/videos/` and embedded below.

In [ ]:
from IPython.display import Video, display
from run_sb3_seeds import record_final_videos

for arm in ["baseline"] + ARMS_FHR:
    for seed, path in record_final_videos(arm):
        print(f"{arm} — seed {seed}: {path.name}")
        display(Video(str(path), embed=True, width=420))